# Tutorial 5-1: Linear Regression (Single and Multivariate)
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

--- 
## Objective
Regression is a form of **supervised learning** where the goal is to predict a continuous numerical value (the response) based on one or more input variables (predictors). In this tutorial, we will explore the math and implementation of linear models:
1. **Simple Linear Regression:** Modeling the relationship between one predictor and one response.
2. **Multivariate Linear Regression:** Extending the model to handle multiple features using vector notation.
3. **Feature Scaling:** Understanding why consistent ranges are vital for interpretable models.
4. **The Normal Equation:** Solving for the optimal parameters analytically using linear algebra.

## 1. Simple Linear Regression: Theory and Manual Calculation
In the simplest case, we assume the relationship between $x$ and $y$ is a straight line:  
$$h_{\theta}(x) = \theta_0 + \theta_1 x_1$$

Where:
- $\theta_0$ is the **intercept** (the value of $y$ when $x=0$).
- $\theta_1$ is the **slope** (the change in $y$ for every unit change in $x$).

Let's use a synthetic dataset representing 'Years of Experience' vs 'Salary' to calculate these parameters manually using the formulas from Slide 7.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Synthetic Data: Years of Experience (X) and Salary in $1000s (y)
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
y = np.array([45, 50, 60, 58, 72, 80, 85, 92, 98, 110])

# Manual Calculation using Slide 7 Formulas
n = len(x)
x_mean = np.mean(x)
y_mean = np.mean(y)

# Calculating Theta_1 (Slope)
numerator = np.sum((x - x_mean) * (y - y_mean))
denominator = np.sum((x - x_mean)**2)
theta_1 = numerator / denominator

# Calculating Theta_0 (Intercept)
theta_0 = y_mean - theta_1 * x_mean

print(f"Manual Intercept (theta_0): {theta_0:.2f}")
print(f"Manual Slope (theta_1): {theta_1:.2f}")

# Visualization
plt.scatter(x, y, color='blue', label='Actual Data')
plt.plot(x, theta_0 + theta_1 * x, color='red', label='Regression Line')
plt.title("Simple Linear Regression")
plt.xlabel("Years of Experience")
plt.ylabel("Salary ($k)")
plt.legend()
plt.show()

## 2. Multivariate Linear Regression
Real-world phenomena are rarely driven by a single factor. In multivariate regression, we use a vector of features $x$ and a vector of weights $\theta$: 
$$h_{\theta}(x) = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + ... + \theta_m x_m$$

We will use the **California Housing Dataset** to predict house values based on multiple features like median income, house age, and average rooms.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression

# Load dataset
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
target = housing.target # Median House Value in $100k

print("Top 5 rows of features:")
display(df.head())

# Fit multivariate model using scikit-learn
model = LinearRegression()
model.fit(df, target)

print(f"\nGlobal Intercept (theta_0): {model.intercept_:.4f}")
print("Coefficients (thetas) for each feature:")
coef_df = pd.DataFrame({'Feature': housing.feature_names, 'Theta': model.coef_})
display(coef_df)

## 3. The Importance of Feature Scaling
In our housing data, 'MedInc' (Median Income) is on a scale of 0–15, while 'Population' ranges into the thousands. As discussed on Slide 12, features with large magnitudes can dominate the model, making it harder to interpret coefficients and often slowing down optimization. 

We will apply **Z-score Normalization** (Standardization):
$$x' = \frac{x - \mu}{\sigma}$$

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=housing.feature_names)

print("Data after Z-score Normalization (Mean=0, Std=1):")
display(df_scaled.describe().round(2).loc[['mean', 'std']])

## 4. The Linear Algebra Solution: Normal Equation
Gradient descent is an iterative approach, but for many problems, we can find the exact solution in one step using the **Normal Equation**:  
$$\theta = (X^T X)^{-1} X^T y$$

To account for the intercept ($\theta_0$), we must prepend a column of 1s to our feature matrix $X$ ($x_0 = 1$).

In [ ]:
# Convert features to matrix and add x0 = 1 column
X = df_scaled.values
X_b = np.c_[np.ones((len(X), 1)), X] # Add intercept column
y = target.reshape(-1, 1)

# Implement the Normal Equation analytically
# Note: inv() computes the matrix inverse
theta_best = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)

print("Optimal Parameters via Normal Equation:")
print(f"Theta_0 (Intercept): {theta_best[0][0]:.4f}")
for i, feature in enumerate(housing.feature_names):
    print(f"Theta_{i+1} ({feature}): {theta_best[i+1][0]:.4f}")

## Summary
We have successfully moved from basic slope-intercept lines to multivariate matrices. Key takeaways:
1. **Relationship Learning:** Linear regression approximates the relationship between inputs and outputs by minimizing the distance between data points and the line.
2. **Vectorization:** Representing features as matrices allows us to compute predictions for thousands of records simultaneously.
3. **Scaling:** Normalizing data ensures that all features contribute proportionally to the model.
4. **Optimization:** While the Normal Equation provides a direct solution, it becomes computationally expensive for very large datasets, which is where we will use **Gradient Descent** in the next tutorial.